# MFA Align
Данный ноутбук посвящен процессу выравнивания набора данных RUSLAN с помощью инструмента Montreal Forced Aligner.

Модели, используемые для выравнивания:
* [russian acoustic model v3.1.0](https://mfa-models.readthedocs.io/en/latest/acoustic/Russian/Russian%20MFA%20acoustic%20model%20v3_1_0.html)
* [russian g2p model v3.1.0](https://mfa-models.readthedocs.io/en/latest/g2p/Russian/Russian%20MFA%20G2P%20model%20v3_1_0.html)
* [russian dictionary v3.1.0](mfa-models.readthedocs.io/en/latest/dictionary/Russian/Russian%20MFA%20dictionary%20v3_1_0.html)  
Также доступны на [huggingface](https://huggingface.co/MontrealCorpusTools/russian_mfa)

In [11]:
import pandas as pd
import json
import glob
import numpy as np
import os

#### Константы: пути к данным и моделям
* RUSLAN_DIR -- место хранения исходных данных -- файла метаданных и аудио
* TEXTGRID_DIR -- место, куда будут сохранены текстовки с разметкой
* MFA_MODELS_DIR -- место для сохранения предобученных и потенциально обученных/доработанных моделей и артефактов от MFA

In [ ]:
RUSLAN_DIR = 'data/RUSLAN'
TEXTGRID_DIR = 'data/RUSLAN_align'
MFA_MODELS_DIR = 'models/mfa'

#### Чтение списка файлов

Файл должен быть результатом нормализации: То есть  в нём помимо двух колонок (идентификатор, сырой текст), должен быть также и нормализованный текст.

In [ ]:
ruslan = pd.read_csv(f'{RUSLAN_ROOT}/metadata_RUSLAN_22200.csv', sep='|', names=['id', 'raw', 'nrm'])

#### Сохранение текстовок для выравнивателя

Необходимо положить файл с расширением .txt рядом с каждым аудифайлом

In [23]:
for i, t in ruslan[['id', 'nrm']].values:
    open(os.path.join(f'{RUSLAN_ROOT}/RUSLAN', i+'.txt'), 'w', encoding='utf-8').write(t)

#### Настройка среды MFA
Предпочтительно через докер, но можно его собрать и в системе

`docker pull mmcauliffe/montreal-forced-aligner:v3.4.1`  
`docker run -it -v $RUSLAN_DIR:/data -v $TEXTGRID_DIR:/align -v $MFA_MODELS_DIR:/models mmcauliffe/montreal-forced-aligner:v3.4.1`  
`mfa model download g2p russian_mfa`  
`mfa model download acoustic russian_mfa`  
`mfa model download dictionary russian_mfa`  
  
`mfa align /data /models/russian_mfa.dict /models/russian_mfa_acoustic /align/v1`

**Важно!** Скачивание моделей через `mfa model` в версии `3.4.1` является устаревшим и планируется к удалению. Можно скачать модели с hugginface или напрямую с сайта. 

Результат работы команды mfa align -- директория с файлами разметки .TextGrid.

#### Проверка результата: потребуется библиотека praatio для работы с TextGrid

In [ ]:
from praatio import textgrid
from prepare_training_data import read_text_grids

In [ ]:
ruslan['phn'] = read_text_grids(ruslan, TEXGRID_DIR)[2]

### Анализ ошибок разметки: поиск Out-Of-Vocabulary слов
Чтение словаря и анализ корпуса на предмет OOV-слов.

Метка `spn` на уровне фонем, а.к. `spoken noise` -- фактор, говорящий о том, что слово не разметилось; одна из причин -- его нет в словаре.

In [193]:
ru_dict = pd.read_csv(f'{MFA_MODELS_DIR/russian_mfa.dict', names=['grapheme', 'p1', 'p2', 'p3', 'p4', 'phoneme'], sep='\t')
ru_dict.loc[ru_dict.phoneme.isna(), 'phoneme'] = ru_dict[ru_dict.phoneme.isna()]['p1']

words = ' '.join(ruslan[ruslan.phn.str.contains('spn')].raw.str.lower().str.replace('[^а-яё ]','', regex=True)).split(' ')
len(words), len(set(words))

dict_words = set(ru_dict.grapheme.unique())

pd.DataFrame({'txt': list(set(words)-dict_words)}).to_csv(f'{RUSLAN_DIR}/oovs_found.txt', index=False, header=False)

#### Предсказание вариантов произношения для OOV-слов с помощью G2P-модели 

`mfa g2p oovs_found.txt /models/russian_mfa_g2p oovs_mfa_g2p_3_1_0.txt`

In [253]:
with open(f'{MFA_MODELS_DIR}/russian_mfa_oov_ruslan.dict', 'w') as writer:
    dict_orig =open(f'{MFA_MODELS_DIR}/russian_mfa.dict').read().strip().split('\n')
    oov_trans = open(f'{RUSLAN_DIR}/oovs_mfa_g2p_3_1_0.txt').read().strip().split('\n')
    writer.write('\n'.join(sorted(dict_orig+oov_trans)))

#### Повторный запуск выравнивания

После пополнения списка OOV-словами, выравнивание должно пойти лучше.

`mfa align /data /models/russian_mfa.dict /models/russian_mfa_acoustic /align/v2`